# Imports


In [1]:
from google.colab import drive
from collections import defaultdict
import time

import numpy as np
from scipy.sparse import csr_matrix

# Data Loading

In [2]:
drive.mount('/content/gdrive')

data_path = 'gdrive/My Drive/ADM_final_assignment/Data/user_movie_rating.npy'


Mounted at /content/gdrive


In [3]:
# Load the file
data = np.load(data_path)

In [4]:
data[:5]

array([[  1,  30,   3],
       [  1, 157,   3],
       [  1, 173,   4],
       [  1, 175,   5],
       [  1, 191,   2]])

In [4]:
users = data[:, 0]
movies = data[:, 1]

In [5]:
movie_user_matrix = csr_matrix((np.ones_like(users), (movies - 1, users - 1)))

# Check the shape
print("Sparse matrix shape (movies x users):", movie_user_matrix.shape)

Sparse matrix shape (movies x users): (17770, 103703)


In [ ]:
# The user 1 rates the movie 30
movie_user_matrix[29,0]

1

# Generating the signatures

In [6]:
def generate_minhash_signature_updated(sparse_matrix, num_permutations, random_seed=None):
    """
    Compute MinHash signatures using permutations.

    Parameters:
        sparse_matrix (csr_matrix): Sparse movie-user matrix where rows represent movies, columns represent users.
        num_permutations (int): Number of MinHash permutations (hash functions) to apply.

    Returns:
        np.ndarray: A 2D array of shape (num_permutations, num_users) where each row
                    corresponds to a permutation and each column to a user's MinHash value.
    """
    if random_seed is not None:
        np.random.seed(random_seed)  # Set the random seed for reproducibility

    num_movies, num_users = sparse_matrix.shape

    # initialize signature matrix with +infinity
    signatures = np.full((num_permutations, num_users), np.inf)

    # generate random hash functions (permutations) for rows
    hash_functions = [np.random.permutation(num_movies) for _ in range(num_permutations)]

    # Precompute inverse hash functions for fast lookup
    inverse_hash_functions = [np.argsort(h) for h in hash_functions]

    # iterate over each row (movie)
    for row_idx in range(num_movies):
        row = sparse_matrix[row_idx, :]  # access the sparse row
        if row.nnz == 0:  # Skip empty rows
            continue

        # Get the users (columns) that rated this movie
        user_indices = row.indices

        for perm_idx in range(num_permutations):  # for each hash function (permutation)
            # Retrieve the position of row_idx in the permutation
            hash_value = inverse_hash_functions[perm_idx][row_idx]
            # Update signatures for all users who rated this movie
            signatures[perm_idx, user_indices] = np.minimum(signatures[perm_idx, user_indices], hash_value)


    return signatures


In [8]:
user_input = input("Enter a random seed (integer): ")
try:
    random_seed = int(user_input)  # Convert user input to integer
except ValueError:
    print("Invalid input! Using a default random seed (42).")

# Generate MinHash signatures
num_permutations = 150
signatures = generate_minhash_signature_updated(movie_user_matrix, num_permutations, random_seed)

Enter a random seed (integer): 42


In [9]:
signatures

array([[  3.,   3.,   0., ...,  20.,  54.,  20.],
       [ 30.,   6.,  10., ...,  10.,  10.,  10.],
       [ 51.,   3.,   3., ..., 160.,  20.,   3.],
       ...,
       [ 76.,  39.,   4., ...,   7.,  37.,  83.],
       [  6.,   1.,   7., ...,   6.,   6.,   6.],
       [ 10.,   1.,   5., ...,  17.,  10.,  10.]])

In [10]:
signatures.max()

853.0

In [ ]:
signatures.min()

0.0

# Locality Sensitive Hashing

In [7]:
def lsh_sorted_buckets(signature_matrix, num_bands, rows_per_band, num_buckets=2**16):
    """
    Perform Locality Sensitive Hashing (LSH) and return sorted buckets by size.

    Parameters:
        signature_matrix (np.ndarray): MinHash signature matrix of shape (n, m),
                                        where n = number of hash functions (total rows),
                                        and m = number of users (columns).
        num_bands (int): Number of bands to divide the matrix into.
        rows_per_band (int): Number of rows per band.
        num_buckets (int): Number of buckets for hashing (default: 2^10).

    Returns:
        list: A list of tuples (bucket_size, bucket_users) sorted by bucket size.
    """
    num_hashes, num_users = signature_matrix.shape
    candidate_buckets = []

    # Process each band
    for band_idx in range(num_bands):
        # Extract rows corresponding to the current band
        start_row = band_idx * rows_per_band
        end_row = min(start_row + rows_per_band, num_hashes)  # Handle shorter signatures
        band = signature_matrix[start_row:end_row, :]

        # Hash columns (users) in the band into buckets
        buckets = defaultdict(list)
        for user_idx in range(num_users):
            band_hash = hash(tuple(band[:, user_idx])) % num_buckets
            buckets[band_hash].append(user_idx)

        # Store each bucket with its size
        for bucket_users in buckets.values():
            candidate_buckets.append((len(bucket_users), bucket_users))

    # Sort buckets by size in ascending order
    candidate_buckets.sort(key=lambda x: x[0])
    return candidate_buckets


In [12]:
candidate_buckets_sorted = lsh_sorted_buckets(signatures, 20, 7, num_buckets=2**16) # check other values
len(candidate_buckets_sorted) # the number of buckets obtained

985272

# Jaccard Similarities Calculation

In [8]:
def compute_jaccard_similarity_sorted(
    sparse_matrix, candidate_buckets, threshold=0.5, max_pairs=200, output_file="output.txt"
):
    """
    Compute Jaccard Similarity for sorted candidate buckets and stop after finding enough pairs.

    Parameters:
        sparse_matrix (csr_matrix): Sparse movie-user matrix where rows represent movies, columns represent users.
        candidate_buckets (list): List of (bucket_size, bucket_users) sorted by size.
        threshold (float): The minimum Jaccard Similarity to include a pair.
        max_pairs (int): Maximum number of pairs to find before stopping.
        output_file (str): File path to write the results.
    """
    sparse_matrix_csc = sparse_matrix.tocsc()
    found_pairs = 0
    checked_pairs = defaultdict(set)

    with open(output_file, "w") as file:
        for _, bucket_users in candidate_buckets:
            # Generate all unique pairs from the bucket
            for i in range(len(bucket_users)):
                for j in range(i + 1, len(bucket_users)):
                    u1, u2 = bucket_users[i], bucket_users[j]

                    # Ensure pair (u1, u2) is ordered as (min, max) to avoid duplicates
                    if u1 > u2:
                        u1, u2 = u2, u1

                    # Skip if already checked
                    if u2 in checked_pairs[u1]:
                        continue
                    checked_pairs[u1].add(u2)

                    # Get the sets of movies rated by user u1 and user u2
                    movies_u1 = sparse_matrix_csc[:, u1]
                    movies_u2 = sparse_matrix_csc[:, u2]

                    # Compute intersection size
                    intersection_size = movies_u1.multiply(movies_u2).nnz
                    union_size = (movies_u1 + movies_u2).nnz

                    if union_size > 0:
                        jaccard_similarity = intersection_size / union_size
                        if jaccard_similarity >= threshold:
                            # Write the pair to the file
                            file.write(f"{u1},{u2}\n")
                            found_pairs += 1

                    # Stop if the desired number of pairs is found
                    if found_pairs >= max_pairs:
                        return found_pairs

In [14]:
compute_jaccard_similarity_sorted(movie_user_matrix, candidate_buckets_sorted, threshold=0.5, output_file="output.txt")

# Ignore this part

## Another way of generating the signatures:
 (more aligned with the pseudo-code from the slides but takes so much time)

In [ ]:
def generate_minhash_signature(sparse_matrix, num_permutations):
    """
    Compute MinHash signatures using permutations.

    Parameters:
        sparse_matrix (csr_matrix): Sparse movie-user matrix where rows represent movies, columns represent users.
        num_permutations (int): Number of MinHash permutations (hash functions) to apply.

    Returns:
        np.ndarray: A 2D array of shape (num_permutations, num_users) where each row
                    corresponds to a permutation and each column to a user's MinHash value.
    """
    num_movies, num_users = sparse_matrix.shape

    # initialize signature matrix with +infinity
    signatures = np.full((num_permutations, num_users), np.inf)

    # generate random hash functions (permutations) for rows
    hash_functions = [np.random.permutation(num_movies) for _ in range(num_permutations)]

    # iterate over each row (movie)
    for row_idx in range(num_movies):
        row = sparse_matrix[row_idx, :]  # access the sparse row
        for user_idx in row.indices:  # for each user (column) with a non-zero value in the row
            for perm_idx in range(num_permutations):  # for each hash function (permutation)
                 # Retrieve the position of row_idx in the permutation
                hash_value = np.where(hash_functions[perm_idx] == row_idx)[0][0]
                if hash_value < signatures[perm_idx, user_idx]:  # Update if hash_value is smaller
                    signatures[perm_idx, user_idx] = hash_value

    return signatures


In [ ]:
signatures_test = generate_minhash_signature(movie_user_matrix, 1)

KeyboardInterrupt: 

In [ ]:
signatures_test

array([[ 10.,  10.,  39., ..., 122.,  83.,  39.]])

In [ ]:
signatures_test.max()

453.0

In [ ]:
signatures_test.min()

0.0

## Another way for Locality Sensitive Hashing:
(it generates candidate pairs directly instead of candidate buckets)

In [ ]:
import numpy as np
from collections import defaultdict

def lsh(signature_matrix, num_bands, rows_per_band, num_buckets=2**10):
    """
    Perform Locality Sensitive Hashing (LSH) on a MinHash signature matrix.

    Parameters:
        signature_matrix (np.ndarray): MinHash signature matrix of shape (n, m),
                                        where n = number of hash functions (total rows),
                                        and m = number of users (columns).
        num_bands (int): Number of bands to divide the matrix into.
        rows_per_band (int): Number of rows per band (b * r must equal n).
        num_buckets (int): Number of buckets for hashing (large value to reduce collisions).

    Returns:
        set: A set of candidate column pairs (as tuples) identified by LSH.
    """
    num_hashes, num_users = signature_matrix.shape
    # this is not necessary, we can accept values for b and r such b*r != number of permutations
    assert num_bands * rows_per_band == num_hashes, \
        "Number of bands * rows per band must equal total hash functions"

    candidate_pairs = set()

    # Process each band
    for band_idx in range(num_bands):
        # Extract rows corresponding to the current band
        start_row = band_idx * rows_per_band
        end_row = start_row + rows_per_band
        band = signature_matrix[start_row:end_row, :]

        # Hash columns (users) in the band into buckets
        buckets = defaultdict(list)
        for user_idx in range(num_users):
            # Use the band as the key for hashing
            # The larger num_backets is, the higher probability of assigning similar pairs to the same bucket
            # But the less candidate pairs we will obtain
            band_hash = hash(tuple(band[:, user_idx])) % num_buckets
            #band_hash = hash(tuple(band[:, user_idx]))
            buckets[band_hash].append(user_idx)

        # Add candidate pairs for columns hashed to the same bucket
        for bucket in buckets.values():
            if len(bucket) > 1:
                # Add all unique pairs from this bucket
                for i in range(len(bucket)):
                    for j in range(i + 1, len(bucket)):
                        candidate_pairs.add((bucket[i], bucket[j]))

    return candidate_pairs


## Another way for Jaccard Similarity
(propreties of sets have been used, all the buckets are processed fully, pairs can be checked many times, so it's not the most optimised way of doing stuff)

In [ ]:
def compute_jaccard_similarity(sparse_matrix, candidate_pairs, threshold=0.5, output_file="output.txt"):
    """
    Compute Jaccard Similarity for candidate pairs and write the results to a file.

    Parameters:
        sparse_matrix (csr_matrix): Sparse movie-user matrix where rows represent movies, columns represent users.
        candidate_pairs (set): Set of candidate pairs as tuples (u1, u2).
        threshold (float): The minimum Jaccard Similarity to include a pair.
        output_file (str): File path to write the results.
    """
    sparse_matrix_csc = sparse_matrix.tocsc()

    with open(output_file, "w") as file:
        for u1, u2 in candidate_pairs:
            # Get the sets of movies rated by user u1 and user u2
            movies_u1 = set(sparse_matrix_csc[:, u1].indices)
            movies_u2 = set(sparse_matrix_csc[:, u2].indices)

            # Compute Jaccard Similarity
            intersection_size = len(movies_u1 & movies_u2)  # Intersection of two sets
            union_size = len(movies_u1 | movies_u2)        # Union of two sets

            if union_size > 0:  # Avoid division by zero
                jaccard_similarity = intersection_size / union_size
                # Check if the similarity meets the threshold
                if jaccard_similarity >= threshold:
                    # Write to the file in the form "u1,u2"
                    file.write(f"{u1},{u2}\n")


## Testing some of the functions using smaller examples to confirm the logic behind them

### Testing the minhash function

In [ ]:
example = np.array([[  1,  1,   3],
       [  1, 3,   3],
       [  2, 2,   4],
       [  2, 4,   5],
       [  3, 1,   2],
       [  3, 2,   2],
       [  3, 3,   2]])

users_example = example[:, 0]
movies_example = example[:, 1]

matrix = csr_matrix((np.ones_like(users_example), (movies_example - 1, users_example - 1)))

signature = generate_minhash_signature_updated(matrix, 1)

[array([0, 2, 3, 1])]


In [ ]:
signature

array([[0., 2., 0.]])

The answer is compatible with the answer obtained manually

### Testing the buckets

In [ ]:
signatures_test = np.array([[  2,  10,   2, 10],
       [  3, 20,   3, 20],
       [  1, 30,   1, 30],
       [  4, 40,  4, 70],
       [  5, 50, 5, 80],
       [  6, 60, 6, 90]])

lsh(signatures_test, 2, 3, num_buckets=2**10)

{(0, 2), (1, 3)}

### Testing the logic behind the 3rd function

In [ ]:
dt = np.array([[  1,  30,   3],
       [  1, 157,   3],
       [  2, 30,   4],
       [  2, 157,   5],
       [  2, 191,   2]])

users_dt = dt[:, 0]
movies_dt = dt[:, 1]

matrix = csr_matrix((np.ones_like(users_dt), (movies_dt - 1, users_dt - 1)))

matrix_csc = matrix.tocsc()
movies_u1 = set(matrix_csc[:, 0].indices)
movies_u2 = set(matrix_csc[:, 1].indices)
intersection_size = len(movies_u1 & movies_u2)  # Intersection of two sets
union_size = len(movies_u1 | movies_u2)        # Union of two sets

if union_size > 0:  # Avoid division by zero
    jaccard_similarity = intersection_size / union_size
    print(jaccard_similarity)

0.6666666666666666


In [ ]:
dt = np.array([[  1,  30,   3],
       [  1, 157,   3],
       [  2, 30,   4],
       [  2, 157,   5],
       [  2, 191,   2]])

users_dt = dt[:, 0]
movies_dt = dt[:, 1]

matrix = csr_matrix((np.ones_like(users_dt), (movies_dt - 1, users_dt - 1)))

matrix_csc = matrix.tocsc()

movies_u1 = matrix_csc[:, 0]
movies_u2 = matrix_csc[:, 1]

# Compute intersection size
intersection_size = movies_u1.multiply(movies_u2).nnz

# Compute union size
union_size = (movies_u1 + movies_u2).nnz

if union_size > 0:  # Avoid division by zero
    jaccard_similarity = intersection_size / union_size
    print(jaccard_similarity)

0.6666666666666666


In [ ]:
movies_u1

{29, 156}

In [ ]:
movies_u2

{29, 156, 190}

In [ ]:
next(iter(candidate_pairs))
52494,99903

(30055, 102021)

In [ ]:
movie_user_matrix_csc = movie_user_matrix.tocsc()

movies_u1 = set(movie_user_matrix_csc[:, 52494].indices)
movies_u2 = set(movie_user_matrix_csc[:, 99903].indices)

# Compute Jaccard Similarity
intersection_size = len(movies_u1 & movies_u2)  # Intersection of two sets
union_size = len(movies_u1 | movies_u2)        # Union of two sets

if union_size > 0:  # Avoid division by zero
    jaccard_similarity = intersection_size / union_size
    print(jaccard_similarity)

0.8378839590443686


In [ ]:
num_backets = 2**16
hash(tuple([2,3,1,1,5,2,10,9,8,14])) % num_backets

63205

In [ ]:
hash(tuple([2,3,1,1,5,2,10,9,8,13])) % num_backets

23059

# Tuning h, b, and r

In [11]:
# Function to tune b, r, and n
def tune_lsh_params(sparse_matrix, similarity_threshold=0.5, max_pairs=200, output_file="output.txt"):
    """
    Tune the LSH parameters b (bands), r (rows per band), and n (number of permutations).

    Parameters:
        sparse_matrix (csr_matrix): Sparse movie-user matrix.
        similarity_threshold (float): Minimum Jaccard Similarity to consider a pair as similar.
        max_pairs (int): Maximum number of similar pairs to find.
        output_file (str): Path to save the output file.

    Returns:
        dict: Results containing parameter configurations and their performance.
    """
    # Define the range of parameters to test
    permutation_options = [100, 150]  # Total number of permutations
    band_options = [10, 15, 20, 25]  # Number of bands (b)

    results = []  # To store performance metrics for each configuration

    for num_permutations in permutation_options:
        for b in band_options:
            if b > num_permutations:
                continue  # Skip invalid configurations where b > n

            # Automatically calculate r as h / b
            r = num_permutations // b  # Rows per band

            # Start timing
            start_time = time.time()

            # Step 1: Generate MinHash signatures
            signatures = generate_minhash_signature_updated(sparse_matrix, num_permutations, 42)

            # Step 2: Perform LSH to get candidate buckets
            candidate_buckets_sorted = lsh_sorted_buckets(signatures, num_bands=b, rows_per_band=r, num_buckets=2**16)

            # Step 3: Compute Jaccard Similarity and write results
            num_pairs = compute_jaccard_similarity_sorted(
                sparse_matrix, candidate_buckets_sorted, threshold=similarity_threshold,
                max_pairs=max_pairs, output_file=output_file
            )

            # Step 4: Measure execution time and count output pairs
            end_time = time.time()
            elapsed_time = end_time - start_time

            # Store the results
            results.append({
                "num_permutations": num_permutations,
                "bands": b,
                "rows_per_band": r,
                "elapsed_time": elapsed_time,
                "num_pairs": num_pairs
            })

            # Print the results for this configuration
            print(f"Permutations: {num_permutations}, Bands: {b}, Rows per Band: {r}, "
      f"Time: {elapsed_time:.2f}s, Pairs: {num_pairs}")

    # Sort results by elapsed time and number of pairs
    results.sort(key=lambda x: (x["elapsed_time"], -x["num_pairs"]))

    return results

# Run the tuning function
results = tune_lsh_params(movie_user_matrix, similarity_threshold=0.5, max_pairs=200, output_file="output.txt")

Permutations: 100, Bands: 5, Rows per Band: 20, Time: 243.60s, Pairs: None
Permutations: 100, Bands: 10, Rows per Band: 10, Time: 382.36s, Pairs: None


KeyboardInterrupt: 